# GhostHunt - JOBS!
### Fake Job Posting Detector
Detects ghost jobs using behavioral signals + NLP on the EMSCAD dataset.

**Stack:** XGBoost · TF-IDF · SHAP · Gradio  
**Dataset:** Employment Scam Aegean Corpus (EMSCAD) — 17,880 job postings, labeled real vs. fake  
**Output:** Ghost probability score (0–100) · SHAP waterfall · Feature risk scorecard

In [1]:
# -- Cell 1: Install all dependencies ---------------------------------------------
!pip install -q xgboost shap gradio plotly scikit-learn pandas numpy


In [2]:
# -- Cell 2: Download EMSCAD dataset ---------------
import urllib.request, os

# Hosted on GitHub
url = "https://raw.githubusercontent.com/Erfaniaa/fake-job-posting-detection/master/dataset.csv"
dest = "fake_job_postings.csv"

if not os.path.exists(dest):
    print("Downloading EMSCAD dataset...")
    urllib.request.urlretrieve(url, dest)
    print(f"Done. Saved to: {dest}")
else:
    print(f"Already downloaded: {dest}")

print(f"File size: {os.path.getsize(dest) / 1024:.1f} KB")


Done. Saved to: fake_job_postings.csv
File size: 48888.3 KB


In [4]:
# -- Cell 3: Loading and inspection ---------------------------------------------------
import pandas as pd
import numpy as np

df = pd.read_csv("fake_job_postings.csv")

print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"Fraudulent distribution:")
print(df["fraudulent"].value_counts())
print(f"Fraud rate: {df["fraudulent"].mean()*100:.1f}%")
df.head(3)


Shape: (17880, 18)
Columns: ['job_id', 'title', 'location', 'department', 'salary_range', 'company_profile', 'description', 'requirements', 'benefits', 'telecommuting', 'has_company_logo', 'has_questions', 'employment_type', 'required_experience', 'required_education', 'industry', 'function', 'fraudulent']
Fraudulent distribution:
fraudulent
0    17014
1      866
Name: count, dtype: int64
Fraud rate: 4.8%


,job_id,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent
0,1,Marketing Intern,"US, NY, New York",Marketing,NaN,"We're Food52, and we've created a groundbreaki...","Food52, a fast-growing, James Beard Award-winn...",Experience with content management systems a m...,NaN,0,1,0,Other,Internship,NaN,NaN,Marketing,0
1,2,Customer Service - Cloud Video Production,"NZ, , Auckland",Success,NaN,"90 Seconds, the worlds Cloud Video Production ...",Organised - Focused - Vibrant - Awesome!Do you...,What we expect from you:Your key responsibilit...,What you will get from usThrough being part of...,0,1,0,Full-time,Not Applicable,NaN,Marketing and Advertising,Customer Service,0
2,3,Commissioning Machinery Assistant (CMA),"US, IA, Wever",NaN,NaN,Valor Services provides Workforce Solutions th...,"Our client, located in Houston, is actively se...",Implement pre-commissioning and commissioning ...,NaN,0,1,0,NaN,NaN,NaN,NaN,NaN,0


In [6]:
# ── Cell 4: Feature Engineering ───────────────────────────────────────────────
import re

def engineer_features(df):
    feat = pd.DataFrame()

    # ── Text fields: fill NaN with empty string
    text_cols = ['title', 'company_profile', 'description', 'requirements', 'benefits']
    for col in text_cols:
        df[col] = df[col].fillna('')

    # ── Combine all text into one field for NLP
    feat['full_text'] = (
        df['title'] + ' ' +
        df['company_profile'] + ' ' +
        df['description'] + ' ' +
        df.get('requirements', '') + ' ' +
        df.get('benefits', '')
    ).str.lower()

    # ── Behavioral / structural features
    feat['has_salary'] = (~df['salary_range'].isna() & (df['salary_range'] != '')).astype(int)
    feat['has_company_logo'] = df.get('has_company_logo', pd.Series(0, index=df.index)).fillna(0).astype(int)
    feat['has_questions'] = df.get('has_questions', pd.Series(0, index=df.index)).fillna(0).astype(int)
    feat['telecommuting'] = df.get('telecommuting', pd.Series(0, index=df.index)).fillna(0).astype(int)

    # Employment type encoded
    emp_map = {'Full-time': 0, 'Part-time': 1, 'Contract': 2, 'Temporary': 3, 'Other': 4, '': 5}
    feat['employment_type_enc'] = df['employment_type'].fillna('').map(emp_map).fillna(5).astype(int)

    # Required experience encoded
    exp_map = {
        'Not Applicable': 0, 'Internship': 1, 'Entry level': 2,
        'Associate': 3, 'Mid-Senior level': 4, 'Director': 5, 'Executive': 6, '': 7
    }
    feat['experience_enc'] = df['required_experience'].fillna('').map(exp_map).fillna(7).astype(int)

    # Required education encoded
    edu_map = {
        'Unspecified': 0, 'High School or equivalent': 1, 'Some College Coursework Completed': 2,
        'Vocational': 3, "Associate's Degree": 4, "Bachelor's Degree": 5,
        "Master's Degree": 6, 'Professional': 7, 'Doctorate': 8, '': 9
    }
    feat['education_enc'] = df['required_education'].fillna('').map(edu_map).fillna(9).astype(int)

    # Function / industry — just flag if missing
    feat['has_function'] = (df['function'].fillna('') != '').astype(int)
    feat['has_industry'] = (df['industry'].fillna('') != '').astype(int)

    # ── NLP-derived features from text
    feat['desc_len'] = df['description'].str.len().fillna(0)
    feat['title_len'] = df['title'].str.len().fillna(0)
    feat['profile_len'] = df['company_profile'].str.len().fillna(0)
    feat['requirements_len'] = df['requirements'].str.len().fillna(0)

    # Suspicious keyword signals
    scam_keywords = [
        'work from home', 'make money', 'earn extra', 'no experience needed',
        'unlimited earning', 'be your own boss', 'financial freedom',
        'join our team today', 'immediate start', 'training provided',
        'weekly pay', 'no degree required', 'flexible hours'
    ]
    feat['scam_keyword_count'] = feat['full_text'].apply(
        lambda t: sum(1 for kw in scam_keywords if kw in t)
    )

    # Vague salary signals
    feat['vague_salary'] = df['salary_range'].fillna('').apply(
        lambda s: 1 if re.search(r'negotiable|competitive|tbd|doe|depends', s.lower()) else 0
    )

    # Exclamation marks (urgency / hype)
    feat['exclamation_count'] = feat['full_text'].str.count(r'!')

    # ALL CAPS words (shouting / scam style)
    feat['caps_word_count'] = feat['full_text'].apply(
        lambda t: len(re.findall(r'\b[A-Z]{3,}\b', t))
    )

    # Email in description (bypass apply process — scam signal)
    feat['has_email_in_desc'] = df['description'].str.contains(
        r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}', regex=True, na=False
    ).astype(int)

    # URL count in description
    feat['url_count'] = df['description'].fillna('').str.count(r'http[s]?://')

    # Location: US-based vs not
    feat['is_us'] = df['location'].fillna('').str.contains('US', case=False).astype(int)

    return feat

features = engineer_features(df.copy())
print(f"Feature matrix shape: {features.shape}")
print(f"\nFeatures: {[c for c in features.columns if c != 'full_text']}")

Feature matrix shape: (17880, 21)

Features: ['has_salary', 'has_company_logo', 'has_questions', 'telecommuting', 'employment_type_enc', 'experience_enc', 'education_enc', 'has_function', 'has_industry', 'desc_len', 'title_len', 'profile_len', 'requirements_len', 'scam_keyword_count', 'vague_salary', 'exclamation_count', 'caps_word_count', 'has_email_in_desc', 'url_count', 'is_us']


In [7]:
# ── Cell 5: TF-IDF on text + combine with behavioral features ─────────────────
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, csr_matrix

# TF-IDF on combined text — top 300 terms
tfidf = TfidfVectorizer(max_features=300, stop_words='english', ngram_range=(1, 2))
X_text = tfidf.fit_transform(features['full_text'])

# Behavioral / structural features (everything except full_text)
behavioral_cols = [c for c in features.columns if c != 'full_text']
X_behavioral = csr_matrix(features[behavioral_cols].values.astype(float))

# Stacking them now
X = hstack([X_behavioral, X_text])
y = df['fraudulent'].values

print(f"Final feature matrix: {X.shape}")
print(f"Label distribution — Real: {(y==0).sum()}, Fake: {(y==1).sum()}")

Final feature matrix: (17880, 320)
Label distribution — Real: 17014, Fake: 866


In [8]:
# ── Cell 6: Train / Test Split + XGBoost ─────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import xgboost as xgb

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Class imbalance: ~5% fraud — use scale_pos_weight
neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
scale = neg / pos
print(f"scale_pos_weight: {scale:.1f}")

model = xgb.XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=scale,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42,
    tree_method='hist',
    device='cuda'   # GPU if available, falls back to CPU
    )

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=100
)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("\n" + "="*50)
print("CLASSIFICATION REPORT")
print("="*50)
print(classification_report(y_test, y_pred, target_names=['Real', 'Fake']))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")

scale_pos_weight: 19.6


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [06:57:20] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[0]	validation_0-logloss:0.67739
[100]	validation_0-logloss:4.33179
[200]	validation_0-logloss:3.64352
[300]	validation_0-logloss:4.92031
[399]	validation_0-logloss:3.83106

CLASSIFICATION REPORT
              precision    recall  f1-score   support

        Real       0.99      0.39      0.56      3403
        Fake       0.07      0.96      0.14       173

    accuracy                           0.42      3576
   macro avg       0.53      0.68      0.35      3576
weighted avg       0.95      0.42      0.54      3576

ROC-AUC: 0.7055


/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [06:57:22] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


In [9]:
# ── Cell 7: SHAP setup (behavioral features only for explainability) ───────────
# We train a second lightweight model on behavioral features only for SHAP
# (SHAP on 300+ TF-IDF features produces unreadable charts)
from sklearn.model_selection import train_test_split
import xgboost as xgb
import shap

X_beh = features[behavioral_cols].values.astype(float)
X_beh_train, X_beh_test, y_beh_train, y_beh_test = train_test_split(
    X_beh, y, test_size=0.2, random_state=42, stratify=y
)

neg2, pos2 = (y_beh_train == 0).sum(), (y_beh_train == 1).sum()

model_beh = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    scale_pos_weight=neg2/pos2,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42,
    tree_method='hist',
    device='cuda'
)
model_beh.fit(X_beh_train, y_beh_train, verbose=False)

# SHAP explainer
explainer = shap.TreeExplainer(model_beh)
print("SHAP explainer ready.")
print(f"Behavioral model AUC: {roc_auc_score(y_beh_test, model_beh.predict_proba(X_beh_test)[:,1]):.4f}")

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [06:57:32] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


SHAP explainer ready.
Behavioral model AUC: 0.9739


In [11]:
# ── Cell 8: Prediction helpers ────────────────────────────────────────────────
import plotly.graph_objects as go
import plotly.io as pio
import re

FEATURE_LABELS = {
    'has_salary': 'Has Salary Range',
    'has_company_logo': 'Has Company Logo',
    'has_questions': 'Has Screening Questions',
    'telecommuting': 'Remote / Telecommuting',
    'employment_type_enc': 'Employment Type Specified',
    'experience_enc': 'Experience Level Specified',
    'education_enc': 'Education Level Specified',
    'has_function': 'Job Function Listed',
    'has_industry': 'Industry Listed',
    'desc_len': 'Description Length',
    'title_len': 'Title Length',
    'profile_len': 'Company Profile Length',
    'requirements_len': 'Requirements Length',
    'scam_keyword_count': 'Scam Keyword Count',
    'vague_salary': 'Vague Salary Language',
    'exclamation_count': 'Exclamation Marks Used',
    'caps_word_count': 'ALL CAPS Words Used',
    'has_email_in_desc': 'Email in Description',
    'url_count': 'URLs in Description',
    'is_us': 'US-Based Location'}

def extract_features_from_text(raw_text):
    """Parse a raw pasted job posting into a feature row."""
    text = raw_text.lower()

    scam_keywords = [
        'work from home', 'make money', 'earn extra', 'no experience needed',
        'unlimited earning', 'be your own boss', 'financial freedom',
        'join our team today', 'immediate start', 'training provided',
        'weekly pay', 'no degree required', 'flexible hours']

    row = {
        'has_salary': 1 if re.search(r'\$[\d,]+|salary|compensation|pay range', text) else 0,
        'has_company_logo': 0,
        'has_questions': 1 if re.search(r'screening|questionnaire|apply.*question|please answer', text) else 0,
        'telecommuting': 1 if re.search(r'remote|work from home|telecommut|wfh', text) else 0,
        'employment_type_enc': 0 if re.search(r'full.?time', text) else (1 if re.search(r'part.?time', text) else 5),
        'experience_enc': 2 if re.search(r'entry.?level|junior|no experience', text) else (4 if re.search(r'senior|mid.?level', text) else 7),
        'education_enc': 5 if re.search(r"bachelor|bs |b\.s\.|b\.a\.", text) else (6 if re.search(r'master|mba|m\.s\.', text) else 9),
        'has_function': 1 if re.search(r'engineering|marketing|sales|finance|operations|design|product', text) else 0,
        'has_industry': 1 if re.search(r'technology|healthcare|finance|retail|media|education|manufacturing', text) else 0,
        'desc_len': len(raw_text),
        'title_len': len(raw_text.split('\n')[0]) if raw_text else 0,
        'profile_len': 0,
        'requirements_len': len(re.findall(r'require|must have|qualif', text)) * 50,
        'scam_keyword_count': sum(1 for kw in scam_keywords if kw in text),
        'vague_salary': 1 if re.search(r'negotiable|competitive|tbd|doe|depends', text) else 0,
        'exclamation_count': raw_text.count('!'),
        'caps_word_count': len(re.findall(r'\b[A-Z]{3,}\b', raw_text)),
        'has_email_in_desc': 1 if re.search(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}', raw_text) else 0,
        'url_count': len(re.findall(r'http[s]?://', raw_text)),
        'is_us': 1 if re.search(r'\bUS\b|united states|USA', raw_text, re.IGNORECASE) else 0}
    return row

def extract_features_from_fields(title, company_profile, description, requirements,
                                  salary_range, employment_type, required_experience,
                                  required_education, has_logo, has_questions,
                                  telecommuting, location, function_, industry):
    """Build feature row from structured form fields."""
    full_text = ' '.join(filter(None, [title, company_profile, description, requirements])).lower()

    scam_keywords = [
        'work from home', 'make money', 'earn extra', 'no experience needed',
        'unlimited earning', 'be your own boss', 'financial freedom',
        'join our team today', 'immediate start', 'training provided',
        'weekly pay', 'no degree required', 'flexible hours']

    emp_map = {'Full-time': 0, 'Part-time': 1, 'Contract': 2, 'Temporary': 3, 'Other': 4, 'Not Specified': 5}
    exp_map = {
        'Not Applicable': 0, 'Internship': 1, 'Entry level': 2,
        'Associate': 3, 'Mid-Senior level': 4, 'Director': 5, 'Executive': 6, 'Not Specified': 7}
    edu_map = {
        'Unspecified': 0, 'High School': 1, 'Some College': 2, 'Vocational': 3,
        "Associate's": 4, "Bachelor's": 5, "Master's": 6, 'Professional': 7, 'Doctorate': 8, 'Not Specified': 9}

    row = {
        'has_salary': 1 if salary_range and salary_range.strip() else 0,
        'has_company_logo': 1 if has_logo else 0,
        'has_questions': 1 if has_questions else 0,
        'telecommuting': 1 if telecommuting else 0,
        'employment_type_enc': emp_map.get(employment_type, 5),
        'experience_enc': exp_map.get(required_experience, 7),
        'education_enc': edu_map.get(required_education, 9),
        'has_function': 1 if function_ and function_.strip() else 0,
        'has_industry': 1 if industry and industry.strip() else 0,
        'desc_len': len(description or ''),
        'title_len': len(title or ''),
        'profile_len': len(company_profile or ''),
        'requirements_len': len(requirements or ''),
        'scam_keyword_count': sum(1 for kw in scam_keywords if kw in full_text),
        'vague_salary': 1 if re.search(r'negotiable|competitive|tbd|doe|depends', (salary_range or '').lower()) else 0,
        'exclamation_count': full_text.count('!'),
        'caps_word_count': len(re.findall(r'\b[A-Z]{3,}\b', ' '.join(filter(None, [title, description])))),
        'has_email_in_desc': 1 if re.search(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}', description or '') else 0,
        'url_count': len(re.findall(r'http[s]?://', description or '')),
        'is_us': 1 if re.search(r'\bUS\b|united states|USA', location or '', re.IGNORECASE) else 0}
    return row

def predict_and_explain(feature_row):
    """Run both models and return ghost score + SHAP values."""
    X_beh_input = np.array([[feature_row[c] for c in behavioral_cols]], dtype=float)
    X_tfidf_input = tfidf.transform([''])  # empty text fallback for text tab
    X_full_input = hstack([csr_matrix(X_beh_input), X_tfidf_input])

    ghost_prob = model.predict_proba(X_full_input)[0][1]
    ghost_score = int(round(ghost_prob * 100))

    shap_values = explainer.shap_values(X_beh_input)
    if isinstance(shap_values, list):
        sv = shap_values[1][0]
    else:
        sv = shap_values[0]

    return ghost_score, sv, X_beh_input[0]


def predict_from_text(feature_row, raw_text):
    """Running models using raw text for TF-IDF."""
    X_beh_input = np.array([[feature_row[c] for c in behavioral_cols]], dtype=float)
    X_tfidf_input = tfidf.transform([raw_text.lower()])
    X_full_input = hstack([csr_matrix(X_beh_input), X_tfidf_input])

    ghost_prob = model.predict_proba(X_full_input)[0][1]
    ghost_score = int(round(ghost_prob * 100))

    shap_values = explainer.shap_values(X_beh_input)
    if isinstance(shap_values, list):
        sv = shap_values[1][0]
    else:
        sv = shap_values[0]

    return ghost_score, sv, X_beh_input[0]

print("Prediction helpers ready.")

Prediction helpers ready.


In [16]:
# ── Cell 9: Chart builders ────────────────────────────────────────────────────
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = 'browser'

def make_gauge(ghost_score):
    color = '#22c55e' if ghost_score < 30 else ('#f59e0b' if ghost_score < 65 else '#ef4444')
    label = 'LIKELY REAL' if ghost_score < 30 else ('SUSPICIOUS' if ghost_score < 65 else 'GHOST JOB')

    fig = go.Figure(go.Indicator(
        mode='gauge+number',
        value=ghost_score,
        number={'suffix': '', 'font': {'size': 48, 'color': color}},
        title={'text': f'Ghost Score  |  {label}', 'font': {'size': 18, 'color': '#e2e8f0'}},
        gauge={
            'axis': {'range': [0, 100], 'tickcolor': '#94a3b8', 'tickfont': {'color': '#94a3b8'}},
            'bar': {'color': color, 'thickness': 0.25},
            'bgcolor': '#1e293b',
            'borderwidth': 0,
            'steps': [
                {'range': [0, 30], 'color': '#14532d'},
                {'range': [30, 65], 'color': '#78350f'},
                {'range': [65, 100], 'color': '#7f1d1d'}
            ],
            'threshold': {
                'line': {'color': color, 'width': 4},
                'thickness': 0.75,
                'value': ghost_score
            }
        }
    ))
    fig.update_layout(
        paper_bgcolor='#0f172a',
        font_color='#e2e8f0',
        height=280,
        margin=dict(t=60, b=20, l=40, r=40))
    return fig


def make_shap_waterfall(shap_vals, feature_values):
    labels = [FEATURE_LABELS.get(c, c) for c in behavioral_cols]

    # Sort by absolute SHAP impact, top 12
    idxs = np.argsort(np.abs(shap_vals))[::-1][:12]
    top_labels = [labels[i] for i in idxs]
    top_shap = [shap_vals[i] for i in idxs]
    top_vals = [feature_values[i] for i in idxs]

    # Reverse for horizontal display (most impactful at top)
    top_labels = top_labels[::-1]
    top_shap = top_shap[::-1]
    top_vals = top_vals[::-1]

    colors = ['#ef4444' if v > 0 else '#22c55e' for v in top_shap]
    hover = [f'Feature value: {top_vals[i]:.2f}<br>SHAP: {top_shap[i]:+.3f}' for i in range(len(top_shap))]

    fig = go.Figure(go.Bar(
        x=top_shap,
        y=top_labels,
        orientation='h',
        marker_color=colors,
        hovertext=hover,
        hoverinfo='text'))
    fig.add_vline(x=0, line_width=1, line_color='#475569')
    fig.update_layout(
        title={'text': 'Feature Impact (red = pushes toward fake, green = pushes toward real)', 'font': {'size': 13, 'color': '#94a3b8'}},
        paper_bgcolor='#0f172a',
        plot_bgcolor='#1e293b',
        font_color='#e2e8f0',
        height=380,
        margin=dict(t=50, b=20, l=220, r=40),
        xaxis=dict(title='SHAP Value', gridcolor='#334155', zerolinecolor='#475569'),
        yaxis=dict(gridcolor='#334155')
    )
    return fig


def make_scorecard(ghost_score, feature_row, shap_vals):
    labels = [FEATURE_LABELS.get(c, c) for c in behavioral_cols]

    rows_html = ''
    idxs = np.argsort(np.abs(shap_vals))[::-1][:10]
    for i in idxs:
        label = labels[i]
        val = feature_row[i]
        sv = shap_vals[i]
        risk_color = '#ef4444' if sv > 0.02 else ('#22c55e' if sv < -0.02 else '#94a3b8')
        risk_label = 'HIGH RISK' if sv > 0.05 else ('RISK' if sv > 0.02 else ('SAFE' if sv < -0.02 else 'NEUTRAL'))
        rows_html += f"""
        <tr>
            <td style='padding:8px 12px;color:#e2e8f0;'>{label}</td>
            <td style='padding:8px 12px;color:#94a3b8;text-align:center;'>{val:.0f}</td>
            <td style='padding:8px 12px;text-align:center;'>
                <span style='color:{risk_color};font-weight:600;font-size:11px;'>{risk_label}</span>
            </td>
        </tr>"""

    verdict_color = '#22c55e' if ghost_score < 30 else ('#f59e0b' if ghost_score < 65 else '#ef4444')
    verdict = 'Likely Real' if ghost_score < 30 else ('Suspicious' if ghost_score < 65 else 'Ghost Job')

    html = f"""
    <div style='background:#0f172a;border-radius:12px;padding:20px;font-family:system-ui,sans-serif;'>
        <div style='display:flex;align-items:center;justify-content:space-between;margin-bottom:16px;'>
            <span style='color:#94a3b8;font-size:13px;'>VERDICT</span>
            <span style='color:{verdict_color};font-size:22px;font-weight:700;'>{verdict}</span>
        </div>
        <table style='width:100%;border-collapse:collapse;'>
            <thead>
                <tr style='border-bottom:1px solid #334155;'>
                    <th style='padding:8px 12px;color:#64748b;font-weight:500;text-align:left;font-size:12px;'>FEATURE</th>
                    <th style='padding:8px 12px;color:#64748b;font-weight:500;text-align:center;font-size:12px;'>VALUE</th>
                    <th style='padding:8px 12px;color:#64748b;font-weight:500;text-align:center;font-size:12px;'>SIGNAL</th>
                </tr>
            </thead>
            <tbody>{rows_html}</tbody>
        </table>
    </div>
    """
    return html

print("Chart builders ready.")

Chart builders ready.


In [17]:
# ── Cell 10: Gradio UI ────────────────────────────────────────────────────────
import gradio as gr

# ── Tab 1: Raw text paste handler
def analyze_text(raw_text):
    if not raw_text or not raw_text.strip():
        empty_fig = go.Figure()
        empty_fig.update_layout(paper_bgcolor='#0f172a', plot_bgcolor='#0f172a', height=280)
        return empty_fig, empty_fig, '<p style="color:#64748b;padding:20px;">Paste a job posting above to analyze it.</p>'

    feature_row = extract_features_from_text(raw_text)
    ghost_score, shap_vals, feat_vals = predict_from_text(feature_row, raw_text)

    gauge_fig = make_gauge(ghost_score)
    shap_fig = make_shap_waterfall(shap_vals, feat_vals)
    scorecard_html = make_scorecard(ghost_score, feat_vals, shap_vals)

    return gauge_fig, shap_fig, scorecard_html


# ── Tab 2: Structured fields handler
def analyze_fields(title, company_profile, description, requirements, salary_range,
                   employment_type, required_experience, required_education,
                   has_logo, has_questions, telecommuting, location, function_, industry):
    feature_row = extract_features_from_fields(
        title, company_profile, description, requirements, salary_range,
        employment_type, required_experience, required_education,
        has_logo, has_questions, telecommuting, location, function_, industry)
    ghost_score, shap_vals, feat_vals = predict_and_explain(feature_row)

    gauge_fig = make_gauge(ghost_score)
    shap_fig = make_shap_waterfall(shap_vals, feat_vals)
    scorecard_html = make_scorecard(ghost_score, feat_vals, shap_vals)

    return gauge_fig, shap_fig, scorecard_html

# ── UI
DARK = '#0f172a'
CARD = '#1e293b'

css = """
body, .gradio-container { background: #0f172a !important; }
.gr-tab-item { color: #94a3b8 !important; }
.gr-tab-item.selected { color: #f1f5f9 !important; border-color: #6366f1 !important; }
label, .gr-form label { color: #94a3b8 !important; }
input, textarea, select { background: #1e293b !important; color: #e2e8f0 !important; border-color: #334155 !important; }
.gr-button-primary { background: #6366f1 !important; border: none !important; }
.gr-button-primary:hover { background: #4f46e5 !important; }
"""

with gr.Blocks(theme=gr.themes.Base(), css=css, title='GhostHunt') as demo:

    gr.HTML("""
    <div style='text-align:center;padding:28px 0 8px;'>
        <div style='font-size:32px;font-weight:800;color:#f1f5f9;letter-spacing:-0.5px;'>GhostHunt</div>
        <div style='font-size:14px;color:#64748b;margin-top:6px;'>Fake Job Posting Detector  |  XGBoost + SHAP  |  EMSCAD Dataset</div>
    </div>
    """)

    with gr.Tabs():

        # ── Tab 1: Paste text
        with gr.TabItem('Paste Job Posting'):
            with gr.Row():
                with gr.Column(scale=1):
                    text_input = gr.Textbox(
                        label='Paste the full job posting here',
                        placeholder='Copy and paste any job posting — title, description, requirements, everything...',
                        lines=14)
                    analyze_text_btn = gr.Button('Analyze', variant='primary')

                with gr.Column(scale=1):
                    gauge_out_t = gr.Plot(label='Ghost Score')
                    scorecard_out_t = gr.HTML(label='Risk Scorecard')

            shap_out_t = gr.Plot(label='Feature Impact (SHAP)')

            analyze_text_btn.click(
                fn=analyze_text,
                inputs=[text_input],
                outputs=[gauge_out_t, shap_out_t, scorecard_out_t])

        # ── Tab 2: Structured fields
        with gr.TabItem('Fill In Fields'):
            with gr.Row():
                with gr.Column(scale=1):
                    f_title = gr.Textbox(label='Job Title', placeholder='e.g. Senior Data Engineer')
                    f_company_profile = gr.Textbox(label='Company Profile', lines=3, placeholder='About the company...')
                    f_description = gr.Textbox(label='Job Description', lines=5, placeholder='Role responsibilities...')
                    f_requirements = gr.Textbox(label='Requirements', lines=3, placeholder='Required skills and qualifications...')
                    f_salary = gr.Textbox(label='Salary Range', placeholder='e.g. $80,000 - $100,000 or leave blank')
                    f_location = gr.Textbox(label='Location', placeholder='e.g. New York, US')

                with gr.Column(scale=1):
                    f_employment_type = gr.Dropdown(
                        label='Employment Type',
                        choices=['Full-time', 'Part-time', 'Contract', 'Temporary', 'Other', 'Not Specified'],
                        value='Not Specified')
                    f_experience = gr.Dropdown(
                        label='Required Experience',
                        choices=['Not Applicable', 'Internship', 'Entry level', 'Associate',
                                 'Mid-Senior level', 'Director', 'Executive', 'Not Specified'],
                        value='Not Specified')
                    f_education = gr.Dropdown(
                        label='Required Education',
                        choices=['Unspecified', 'High School', 'Some College', 'Vocational',
                                 "Associate's", "Bachelor's", "Master's", 'Professional', 'Doctorate', 'Not Specified'],
                        value='Not Specified')
                    f_function = gr.Textbox(label='Job Function', placeholder='e.g. Engineering, Marketing')
                    f_industry = gr.Textbox(label='Industry', placeholder='e.g. Technology, Healthcare')
                    f_logo = gr.Checkbox(label='Has Company Logo')
                    f_questions = gr.Checkbox(label='Has Screening Questions')
                    f_remote = gr.Checkbox(label='Remote / Telecommuting')

                    analyze_fields_btn = gr.Button('Analyze', variant='primary')

            with gr.Row():
                gauge_out_f = gr.Plot(label='Ghost Score')
                scorecard_out_f = gr.HTML(label='Risk Scorecard')

            shap_out_f = gr.Plot(label='Feature Impact (SHAP)')

            analyze_fields_btn.click(
                fn=analyze_fields,
                inputs=[
                    f_title, f_company_profile, f_description, f_requirements,
                    f_salary, f_employment_type, f_experience, f_education,
                    f_logo, f_questions, f_remote, f_location, f_function, f_industry
                ],
                outputs=[gauge_out_f, shap_out_f, scorecard_out_f])

demo.launch(share=True, debug=False)

/tmp/ipykernel_561/1525289055.py:51: DeprecationWarning:

The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.

/tmp/ipykernel_561/1525289055.py:51: DeprecationWarning:

The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.



Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cfdbc54ebb44f9c694.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
